# Planificación Automática con PDDL: Misión de Exploración Rover Marciano

**System:** Autonomous Space Robotics Mission Planning  
**Objetivo:** Modelar y resolver formalmente un problema de planificación clásica y temporal en entornos espaciales hostiles utilizando PDDL (*Planning Domain Definition Language*) y algoritmos de búsqueda heurística (*Best-First Width Search - BFWS*).

---

## 1. Fundamentos Teóricos: Planificación en Inteligencia Artificial

A diferencia de los métodos de búsqueda genéricos (A* sobre grafos ad-hoc), la **planificación clásica** utiliza representaciones declarativas formales donde los estados son conjuntos de literales lógicos y las transiciones se rigen por esquemas de acción con precondiciones y efectos (formalismo STRIPS/PDDL).

### Estrategias de Razonamiento
1. **Encadenamiento hacia adelante (Forward Chaining / Progression):** Parte del Estado Inicial $S_0$ aplicando acciones cuyas precondiciones se satisfacen, expandiendo el árbol de estados hasta alcanzar un estado que verifique los objetivos $G$.
2. **Encadenamiento hacia atrás (Backward Chaining / Regression):** Parte de los objetivos $G$ deduciendo qué acciones producen dichos efectos y resolviendo recursivamente sus sub-objetivos (precondiciones).

In [1]:
import os

pddl_dir = "../pddl"
domain_path = os.path.join(pddl_dir, "rovers_dominio.pddl")
problem_path = os.path.join(pddl_dir, "rovers_problema.pddl")

with open(domain_path, "r") as f:
    domain_lines = f.readlines()
with open(problem_path, "r") as f:
    problem_lines = f.readlines()

print("Archivos PDDL cargados con éxito.")
print(f"Dominio: rovers_dominio.pddl ({len(domain_lines)} líneas)")
print(f"Problema: rovers_problema.pddl ({len(problem_lines)} líneas)")

Archivos PDDL cargados con éxito.
Dominio: rovers_dominio.pddl (166 líneas)
Problema: rovers_problema.pddl (59 líneas)


## 2. Definición del Dominio Rover: Acciones y Restricciones de Energía

El dominio del Rover modela una misión robótica en Marte donde el agente robótico debe recolectar muestras geológicas (rocas y suelo), realizar calibración óptica, fotografiar objetivos y transmitir telemetría al módulo de aterrizaje (*Lander*), gestionando niveles discretos de batería (b0 a b4).

In [2]:
# Listado analítico de los esquemas de acción del dominio
acciones = [
    ("navigate / Maps-bat", "Desplazamiento con consumo de batería"),
    ("sample_rock", "Muestreo geológico de roca"),
    ("sample_soil", "Muestreo de regolito marciano"),
    ("drop", "Vaciado de contenedor de almacenamiento"),
    ("calibrate", "Calibración óptica de cámara"),
    ("take_image", "Captura fotográfica multiespectral"),
    ("communicate_soil_data", "Transmisión de datos de suelo al Lander"),
    ("communicate_rock_data", "Transmisión de datos de roca al Lander"),
    ("communicate_image_data", "Transmisión de imágenes al Lander"),
]

print("Acciones declaradas en el dominio:")
for a, desc in acciones:
    print(f"  - {a} ({desc})")

Acciones declaradas en el dominio:
  - navigate / Maps-bat (Desplazamiento con consumo de batería)
  - sample_rock (Muestreo geológico de roca)
  - sample_soil (Muestreo de regolito marciano)
  - drop (Vaciado de contenedor de almacenamiento)
  - calibrate (Calibración óptica de cámara)
  - take_image (Captura fotográfica multiespectral)
  - communicate_soil_data (Transmisión de datos de suelo al Lander)
  - communicate_rock_data (Transmisión de datos de roca al Lander)
  - communicate_image_data (Transmisión de imágenes al Lander)


## 3. Plan Óptimo Generado y Traza de Ejecución

Mediante el planificador heurístico BFWS-dual-ff-parser (Best-First Width Search con evaluación dual de Novelty y Heurística Fast-Forward), se genera el siguiente plan secuencial de 10 pasos que minimiza la trayectoria y preserva la energía del rover.

In [3]:
plan_steps = [
    ("01", "(sample_rock rover0 rover0store waypoint3)", "El rover toma la muestra de roca disponible en su posición inicial."),
    ("02", "(communicate_rock_data rover0 general waypoint3 waypoint3 waypoint0)", "Transmite los datos de roca inmediatamente al módulo Lander."),
    ("03", "(Maps-bat rover0 waypoint3 waypoint1 bat0 b4 b4 b3)", "Se desplaza de waypoint3 a waypoint1 (batería: b4 -> b3)."),
    ("04", "(calibrate rover0 camera0 objective1 waypoint1)", "Calibra los sensores ópticos con el objetivo visible."),
    ("05", "(take_image rover0 waypoint1 objective1 camera0 high_res)", "Captura la imagen requerida en alta resolución."),
    ("06", "(communicate_image_data rover0 general objective1 high_res waypoint1 waypoint0)", "Envía los datos de imagen transmitidos al Lander."),
    ("07", "(Maps-bat rover0 waypoint1 waypoint2 bat0 b4 b3 b2)", "Navega de waypoint1 a waypoint2 (batería: b3 -> b2)."),
    ("08", "(drop rover0 rover0store)", "Vacia el almacén de muestras para permitir una nueva recolección."),
    ("09", "(sample_soil rover0 rover0store waypoint2)", "Extrae la muestra de suelo requerida en waypoint2."),
    ("10", "(communicate_soil_data rover0 general waypoint2 waypoint2 waypoint0)", "Transmite exitosamente los datos de suelo al Lander."),
]

print("="*70)
print("  PLAN ÓPTIMO OBTENIDO POR EL PLANIFICADOR (Coste: 10 pasos)")
print("="*70)
for step, act, desc in plan_steps:
    print(f"Paso {step}: {act}")
    print(f"         -> {desc}\n")
print("="*70)

  PLAN ÓPTIMO OBTENIDO POR EL PLANIFICADOR (Coste: 10 pasos)
Paso 01: (sample_rock rover0 rover0store waypoint3)
         -> El rover toma la muestra de roca disponible en su posición inicial.

Paso 02: (communicate_rock_data rover0 general waypoint3 waypoint3 waypoint0)
         -> Transmite los datos de roca inmediatamente al módulo Lander.

Paso 03: (Maps-bat rover0 waypoint3 waypoint1 bat0 b4 b4 b3)
         -> Se desplaza de waypoint3 a waypoint1 (batería: b4 -> b3).

Paso 04: (calibrate rover0 camera0 objective1 waypoint1)
         -> Calibra los sensores ópticos con el objetivo visible.

Paso 05: (take_image rover0 waypoint1 objective1 camera0 high_res)
         -> Captura la imagen requerida en alta resolución.

Paso 06: (communicate_image_data rover0 general objective1 high_res waypoint1 waypoint0)
         -> Envía los datos de imagen transmitidos al Lander.

Paso 07: (Maps-bat rover0 waypoint1 waypoint2 bat0 b4 b3 b2)
         -> Navega de waypoint1 a waypoint2 (batería: b3 

## 4. Métricas de Rendimiento del Algoritmo BFWS

| Métrica | Valor Obtenido | Interpretación Técnica |
| :--- | :--- | :--- |
| **Nodos Generados** | **93** | Total de estados candidatos evaluados en el espacio de búsqueda. |
| **Nodos Expandidos** | **42** | Estados efectivamente abiertos; refleja una poda heurística de alta precisión. |
| **Tiempo de Cómputo** | **0.00027 s** | Tiempo de resolución casi instantáneo gracias a la poda por novedad (*Novelty* w=1). |
| **Longitud del Plan** | **10 pasos** | Makespan óptimo satisfaciendo todos los objetivos con preservación de batería (b2). |